# 14b — local Phase3 comparison

This notebook runs on your computer and defaults to the post-review wildfire revision. Set CONFIG to local_config.json to inspect the original experiment. Its old filename is retained for compatibility.

B0 uses **p2 from phase2_generation_prompts.yaml**. B1 uses structured abstention; B2 is a development-calibrated gate derived from B1, with no extra API calls. No RAGAS or LLM judge.

Use your current key through the hidden prompt below or GEMINI_API_KEY. It is not written to files. User ceilings are 15 requests/minute and 500/day; retries count, and the default safety reserve stops at 475 local attempts. This revision reuses 394 exact historical requests and generates six changed responses. Leave EXECUTE=False to display saved results without a key.


In [ ]:
from pathlib import Path
import sys
import json

# Works when launched from the repository, Phase3, or a directory below either.
locations = [Path.cwd(), *Path.cwd().parents]
PHASE3 = next((p if (p / "phase3_run.py").is_file() else p / "Phase3"
               for p in locations
               if (p / "phase3_run.py").is_file() or (p / "Phase3/phase3_run.py").is_file()), None)
if PHASE3 is None:
    raise RuntimeError("Open this notebook from the project or Phase3 directory.")
sys.path.insert(0, str(PHASE3))
from phase3_run import load_settings, preflight
CONFIG = "revision_config.json"  # local_config.json selects the original experiment
settings = load_settings(PHASE3 / CONFIG)
print("Kernel:", sys.executable)
print("Phase3:", PHASE3)


## 1. Read-only readiness check

Generation requires the repaired data and the actual contexts to be checked first. Missing local packages/weights or unresolved labels are reported without downloading anything or contacting Gemini.


In [ ]:
report = preflight(settings, "smoke")
print(json.dumps(report, indent=2))


## 2. Run when ready

Leave EXECUTE=False to inspect safely. With EXECUTE=True, choose all or one stage. Interrupting and rerunning preserves responses and request usage. On HTTP 429/authentication errors, the runner pauses rather than repeatedly sending requests.

Keep api_usage.sqlite: it is shared across stages and run folders. It cannot see requests made by other applications in the same Google project.


In [ ]:
EXECUTE = False
STAGE = "all"  # all, smoke, development, final

if EXECUTE:
    import os
    from getpass import getpass
    from phase3_run import run
    stages = ["smoke", "development", "final"] if STAGE == "all" else [STAGE]
    if any(s not in {"smoke", "development", "final"} for s in stages):
        raise ValueError("Invalid stage")
    readiness = preflight(settings, stages[0])
    if readiness["errors"]:
        raise RuntimeError("\n".join(readiness["errors"]))
    api_key = os.environ.get("GEMINI_API_KEY") or getpass("Current Gemini API key (hidden): ")
    try:
        for stage in stages:
            print(json.dumps(run(settings, stage, api_key), indent=2))
    finally:
        api_key = None
else:
    print("Execution is off; no API key was requested and no requests were sent.")


## 3. Inspect results

Each stage saves raw answers, per-case errors, control EM/token F1, citation-index validity, subtype counts and request usage. A counterfactual correction is flagged for interpretation, not automatically called a hallucination. Final uses development's locked threshold and policy; it never tunes them.


In [ ]:
for stage in ("smoke", "development", "final"):
    result_path = settings["work"] / stage / "results.json"
    if result_path.exists():
        result = json.loads(result_path.read_text(encoding="utf-8"))
        print(stage, json.dumps(result["summaries"], indent=2))
    else:
        print(stage + ": no results yet")
